In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("fused data.csv", encoding="utf-8-sig")

years = list(range(2010, 2025))
cr_cols = [f"Nb CR {y}" for y in years if f"Nb CR {y}" in df.columns]

for c in cr_cols + ["Masse rejetée à chaque CR (kg)"]:
    df[c] = pd.to_numeric(
        df[c].astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )

threshold_cr = 365 / 60

def calc_equipment_leakage(row):
    total = 0.0
    stop_year = None

    for y in years:
        col = f"Nb CR {y}"
        if col not in row.index:
            continue

        nb_cr = row[col]
        mass = row["Masse rejetée à chaque CR (kg)"]

        if pd.isna(nb_cr) or pd.isna(mass):
            continue

        yearly_leak = nb_cr * mass
        total += yearly_leak

        if yearly_leak >= 5 or (365 / nb_cr) <= 60:
            stop_year = y
            break

    return pd.Series({
        "equipment_leakage": total,
        "stop_year": stop_year
    })

result = df.apply(calc_equipment_leakage, axis=1)

output = pd.concat([df[["N Equipement"]], result], axis=1)

total_leakage = output["equipment_leakage"].sum()

output.to_csv(
    "equipment_leakage_stopped_by_threshold.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Total leakage:", total_leakage)
print("Number of CSEMs that triggered criteria:", output["stop_year"].notna().sum())

Total leakage: 17865.02665
Number of CSEMs that triggered criteria: 681


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("fused data.csv", encoding="utf-8-sig")

df.columns = (
    df.columns.astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
    .str.strip()
)

years = list(range(2010, 2025))
cr_cols = [f"Nb CR {y}" for y in years if f"Nb CR {y}" in df.columns]

for c in cr_cols + ["Masse rejetée à chaque CR (kg)"]:
    df[c] = pd.to_numeric(
        df[c].astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )

def calc_row(row):
    total = 0.0
    stop_year = np.nan
    stopped_reason = np.nan

    for y in years:
        col = f"Nb CR {y}"
        if col not in row.index:
            continue

        nb = row[col]
        mass = row["Masse rejetée à chaque CR (kg)"]

        if pd.isna(nb) or pd.isna(mass):
            continue

        leak = nb * mass
        total += leak

        cond1 = leak >= 5
        cond2 = (365 / nb) <= 60

        if cond1 or cond2:
            stop_year = y
            if cond1 and cond2:
                stopped_reason = "environment and exploit"
            elif cond1:
                stopped_reason = "environment"
            else:
                stopped_reason = "exploit"
            break

    return pd.Series({
        "equipment_leakage": total,
        "stop_year": stop_year,
        "stopped_reason": stopped_reason
    })

out = pd.concat([df[["N Equipement"]], df.apply(calc_row, axis=1)], axis=1)

out.to_csv("equipment_leakage_summary_only.csv", index=False, encoding="utf-8-sig")

summary = pd.DataFrame({
    "total_leakage": [out["equipment_leakage"].sum()],
    "equipment_count": [len(out)],
    "stopped_equipment_count": [out["stop_year"].notna().sum()]
})
summary.to_csv("summary_total_leakage.csv", index=False, encoding="utf-8-sig")

print("Total leakage:", out["equipment_leakage"].sum())
print("Number of CSEMs that triggered criteria:", out["stop_year"].notna().sum())
print("stopped_reason counts:\n", out["stopped_reason"].value_counts())
print("stop_year counts:\n", out["stop_year"].value_counts())

Total leakage: 17865.02665
Number of CSEMs that triggered criteria: 681
stopped_reason counts:
 stopped_reason
environment                501
environment and exploit    120
exploit                     60
Name: count, dtype: int64
stop_year counts:
 stop_year
2010.0    158
2011.0     70
2012.0     55
2013.0     51
2014.0     49
2016.0     47
2015.0     46
2018.0     45
2017.0     34
2019.0     29
2021.0     24
2022.0     21
2020.0     20
2024.0     19
2023.0     13
Name: count, dtype: int64
